<a href="https://colab.research.google.com/github/dlsanf2000/2021_KOSMI/blob/main/Colon_cancer_patients_using_ANN_%2B_XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
!pip install -U -q PyDrive 
from pydrive.auth import GoogleAuth 
from pydrive.drive import GoogleDrive 
from google.colab import auth
from oauth2client.client import GoogleCredentials
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
!pip install openpyxl=='3.0.0'

     |████████████████████████████████| 172 kB 4.2 MB/s 
  Created wheel for openpyxl: filename=openpyxl-3.0.0-py2.py3-none-any.whl size=241207 sha256=a59c53a42db10d973e17f74862200e82946f1403be425fe5e70fba56528cd4be
  Stored in directory: /root/.cache/pip/wheels/c7/64/ff/ce98f6e1d2701ae8e216c875da62feed2839ac8a3cae0ab8af
Successfully built openpyxl
  Attempting uninstall: openpyxl
    Found existing installation: openpyxl 3.0.9
    Uninstalling openpyxl-3.0.9:
      Successfully uninstalled openpyxl-3.0.9


In [3]:
df_raw = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/NIPA/dataset/nipa_coloncanceronly.xlsx', header = 2)
df_raw.head()
df_raw = df_raw.drop([0,1])

df1 = df_raw[['Age', 'Sex', 'ASA', 'BMI', 'DM_history', 'Pulmonary_disease','Liver_disease', 'Kidney_disease', 'Heart_disease', 'Initial_CEA', 'Hereditary_colorectal_tumor', 'Primary_recurrent', 
              'Perforation', 'Obstruction', 'Emergency', 'Prior_Dx_cancer','pT','pM', 'pTNM', 'K-ras', 'N-ras', 'BRAF', 'LVI', 'PNI', 'pN', 
              'Harvested_LN','Positive_LN','Early_Complication', 'Postop_Chemotherapy', 'Postop_Chemo_Regimen', 'Survival_Status', 'Death_Date', 'Last_Follow_Up_Date', 'Operation_date']]
df = df1.copy()

In [4]:
df

,Age,Sex,ASA,BMI,DM_history,Pulmonary_disease,Liver_disease,Kidney_disease,Heart_disease,Initial_CEA,...,pN,Harvested_LN,Positive_LN,Early_Complication,Postop_Chemotherapy,Postop_Chemo_Regimen,Survival_Status,Death_Date,Last_Follow_Up_Date,Operation_date
2,69,1,3,22.86,1,0,0,0,0,0.5,...,31,33,1,0,1,1,1,2007-09-24 00:00:00,2007-09-24 00:00:00,2004-01-08 00:00:00
3,55,2,2,28.04,0,0,0,0,1,0.03,...,2,13,0,0,1,1,0,NaN,2019-03-07 00:00:00,2004-01-15 00:00:00
4,48,1,3,24.44,0,0,0,0,0,NaN,...,2,58,0,0,1,1,1,2018-12-04 00:00:00,2018-12-04 00:00:00,2004-01-20 00:00:00
5,65,1,2,17.16,0,0,0,0,0,0.5,...,2,27,0,0,1,1,0,NaN,2019-09-05 00:00:00,2004-01-24 00:00:00
6,54,2,3,18.82,0,0,0,0,1,26.36,...,2,28,0,0,1,1,0,NaN,2019-12-05 00:00:00,2004-01-26 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2088,75,1,2,18.52,0,1,0,0,"1, 5",3.2,...,42,40,28,0,0,NaN,0,NaN,2019-04-22 00:00:00,2018-12-17 00:00:00
2089,77,2,2,21.7,0,0,0,0,"1, 3",1.65,...,2,39,0,0,1,3,0,NaN,2019-10-25 00:00:00,2018-12-18 00:00:00
2090,66,1,2,31.81,1,0,0,1,1,175.73,...,42,15,7,0,1,3 + 7,0,NaN,2019-11-06 00:00:00,2018-12-19 00:00:00
2091,69,2,2,26.77,1,0,0,0,1,52.57,...,2,16,0,0,0,NaN,0,NaN,2019-01-22 00:00:00,2018-12-21 00:00:00


## 환자 삭제 및 대체

In [5]:
#Initial_CEA 대체 
df['Initial_CEA'] = df['Initial_CEA'].replace(9999, df['Initial_CEA'].median())
df['Initial_CEA'] = df['Initial_CEA'].fillna(df['Initial_CEA'].median())

In [6]:
df['Initial_CEA'].isna().sum()

0

In [7]:
print(df['Primary_recurrent'].value_counts())

1    2081
3      10
Name: Primary_recurrent, dtype: int64


In [8]:
print(df['Hereditary_colorectal_tumor'].value_counts())

0     2063
12      25
11       3
Name: Hereditary_colorectal_tumor, dtype: int64


In [9]:
#유전성 대장암 제외 - 28건

df = df[df['Hereditary_colorectal_tumor'] != 11]
df = df[df['Hereditary_colorectal_tumor'] != 12]

In [10]:
print(df['Perforation'].value_counts())

0       1546
9999     316
1        201
Name: Perforation, dtype: int64


In [11]:
#Perforation 대체 - 320건
df['Perforation'] = df['Perforation'].replace(9999, 0)

In [12]:
#Obstruction 대체 - 4건
df['Obstruction'] = df['Obstruction'].replace(9999, 0)

In [13]:
print(df['Emergency'].value_counts())

1    1964
2      99
Name: Emergency, dtype: int64


In [14]:
#Emergency 환자 제외 - 99건

df = df[df['Emergency'] != 2]

In [15]:
#Harvested_LN nodata 삭제

df['Harvested_LN']=df['Harvested_LN'].fillna(9999)
df = df[df['Harvested_LN'] != 9999]

In [16]:
#Positive_LN nodata 삭제

df['Positive_LN']=df['Positive_LN'].fillna(9999)
df = df[df['Positive_LN'] != 9999]

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until


In [17]:
# Early_Complication nodata 삭제

df['Early_Complication']=df['Early_Complication'].fillna(9999)
df = df[df['Early_Complication'] != 9999]

In [18]:
#LVI & PNI

#1 LVI의 (필드값없음)을 3(not assessed), PNI의 (필드값없음)을 4(not assessed)로 적용
df['LVI'] = df['LVI'].fillna(3)
df['PNI'] = df['PNI'].fillna(4)


#2 변수 둘 다 삭제

#3 LVI만 넣어보기 (이 때, LVI의 3, 필드값없음 54건을 삭제)
#df['LVI']=df['LVI'].fillna(9999)
#df = df[df['LVI'] != '9999']

In [19]:
df['Positive_LN'].isna().sum()

0

In [20]:
df

,Age,Sex,ASA,BMI,DM_history,Pulmonary_disease,Liver_disease,Kidney_disease,Heart_disease,Initial_CEA,...,pN,Harvested_LN,Positive_LN,Early_Complication,Postop_Chemotherapy,Postop_Chemo_Regimen,Survival_Status,Death_Date,Last_Follow_Up_Date,Operation_date
2,69,1,3,22.86,1,0,0,0,0,0.50,...,31,33,1,0,1,1,1,2007-09-24 00:00:00,2007-09-24 00:00:00,2004-01-08 00:00:00
3,55,2,2,28.04,0,0,0,0,1,0.03,...,2,13,0,0,1,1,0,NaN,2019-03-07 00:00:00,2004-01-15 00:00:00
5,65,1,2,17.16,0,0,0,0,0,0.50,...,2,27,0,0,1,1,0,NaN,2019-09-05 00:00:00,2004-01-24 00:00:00
6,54,2,3,18.82,0,0,0,0,1,26.36,...,2,28,0,0,1,1,0,NaN,2019-12-05 00:00:00,2004-01-26 00:00:00
7,56,1,2,18.94,0,0,0,0,0,4.78,...,2,33,0,0,1,1,0,NaN,2018-12-31 00:00:00,2004-01-27 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2087,50,1,2,21.72,1,0,0,0,0,4.29,...,41,21,4,0,1,4,0,NaN,2019-10-02 00:00:00,2018-12-17 00:00:00
2088,75,1,2,18.52,0,1,0,0,"1, 5",3.20,...,42,40,28,0,0,NaN,0,NaN,2019-04-22 00:00:00,2018-12-17 00:00:00
2089,77,2,2,21.7,0,0,0,0,"1, 3",1.65,...,2,39,0,0,1,3,0,NaN,2019-10-25 00:00:00,2018-12-18 00:00:00
2090,66,1,2,31.81,1,0,0,1,1,175.73,...,42,15,7,0,1,3 + 7,0,NaN,2019-11-06 00:00:00,2018-12-19 00:00:00


In [21]:
df.isna().sum()

Age                               0
Sex                               0
ASA                               0
BMI                               0
DM_history                        0
Pulmonary_disease                 0
Liver_disease                     0
Kidney_disease                    0
Heart_disease                     0
Initial_CEA                       0
Hereditary_colorectal_tumor       0
Primary_recurrent                 0
Perforation                       0
Obstruction                       0
Emergency                         0
Prior_Dx_cancer                   0
pT                                3
pM                                3
pTNM                              3
K-ras                             0
N-ras                             0
BRAF                              0
LVI                               0
PNI                               0
pN                                0
Harvested_LN                      0
Positive_LN                       0
Early_Complication          

## 변수 전처리

### 범주화

In [22]:
#Age

Age_cate = [65]
df['Age'] = np.digitize(df['Age'], Age_cate)
df

,Age,Sex,ASA,BMI,DM_history,Pulmonary_disease,Liver_disease,Kidney_disease,Heart_disease,Initial_CEA,...,pN,Harvested_LN,Positive_LN,Early_Complication,Postop_Chemotherapy,Postop_Chemo_Regimen,Survival_Status,Death_Date,Last_Follow_Up_Date,Operation_date
2,1,1,3,22.86,1,0,0,0,0,0.50,...,31,33,1,0,1,1,1,2007-09-24 00:00:00,2007-09-24 00:00:00,2004-01-08 00:00:00
3,0,2,2,28.04,0,0,0,0,1,0.03,...,2,13,0,0,1,1,0,NaN,2019-03-07 00:00:00,2004-01-15 00:00:00
5,1,1,2,17.16,0,0,0,0,0,0.50,...,2,27,0,0,1,1,0,NaN,2019-09-05 00:00:00,2004-01-24 00:00:00
6,0,2,3,18.82,0,0,0,0,1,26.36,...,2,28,0,0,1,1,0,NaN,2019-12-05 00:00:00,2004-01-26 00:00:00
7,0,1,2,18.94,0,0,0,0,0,4.78,...,2,33,0,0,1,1,0,NaN,2018-12-31 00:00:00,2004-01-27 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2087,0,1,2,21.72,1,0,0,0,0,4.29,...,41,21,4,0,1,4,0,NaN,2019-10-02 00:00:00,2018-12-17 00:00:00
2088,1,1,2,18.52,0,1,0,0,"1, 5",3.20,...,42,40,28,0,0,NaN,0,NaN,2019-04-22 00:00:00,2018-12-17 00:00:00
2089,1,2,2,21.7,0,0,0,0,"1, 3",1.65,...,2,39,0,0,1,3,0,NaN,2019-10-25 00:00:00,2018-12-18 00:00:00
2090,1,1,2,31.81,1,0,0,1,1,175.73,...,42,15,7,0,1,3 + 7,0,NaN,2019-11-06 00:00:00,2018-12-19 00:00:00


In [23]:
#BMI

bmi_cate = [0, 18.5, 25]
df['BMI'] = np.digitize(df['BMI'], bmi_cate)
df

,Age,Sex,ASA,BMI,DM_history,Pulmonary_disease,Liver_disease,Kidney_disease,Heart_disease,Initial_CEA,...,pN,Harvested_LN,Positive_LN,Early_Complication,Postop_Chemotherapy,Postop_Chemo_Regimen,Survival_Status,Death_Date,Last_Follow_Up_Date,Operation_date
2,1,1,3,2,1,0,0,0,0,0.50,...,31,33,1,0,1,1,1,2007-09-24 00:00:00,2007-09-24 00:00:00,2004-01-08 00:00:00
3,0,2,2,3,0,0,0,0,1,0.03,...,2,13,0,0,1,1,0,NaN,2019-03-07 00:00:00,2004-01-15 00:00:00
5,1,1,2,1,0,0,0,0,0,0.50,...,2,27,0,0,1,1,0,NaN,2019-09-05 00:00:00,2004-01-24 00:00:00
6,0,2,3,2,0,0,0,0,1,26.36,...,2,28,0,0,1,1,0,NaN,2019-12-05 00:00:00,2004-01-26 00:00:00
7,0,1,2,2,0,0,0,0,0,4.78,...,2,33,0,0,1,1,0,NaN,2018-12-31 00:00:00,2004-01-27 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2087,0,1,2,2,1,0,0,0,0,4.29,...,41,21,4,0,1,4,0,NaN,2019-10-02 00:00:00,2018-12-17 00:00:00
2088,1,1,2,2,0,1,0,0,"1, 5",3.20,...,42,40,28,0,0,NaN,0,NaN,2019-04-22 00:00:00,2018-12-17 00:00:00
2089,1,2,2,2,0,0,0,0,"1, 3",1.65,...,2,39,0,0,1,3,0,NaN,2019-10-25 00:00:00,2018-12-18 00:00:00
2090,1,1,2,3,1,0,0,1,1,175.73,...,42,15,7,0,1,3 + 7,0,NaN,2019-11-06 00:00:00,2018-12-19 00:00:00


In [24]:
#Heart_disease

df['Heart_disease'] = df['Heart_disease'].astype(str)

In [25]:
df['Heart_disease'] = df['Heart_disease'].replace('1, 3', '1')
df['Heart_disease'] = df['Heart_disease'].replace('3', '1')
df['Heart_disease'] = df['Heart_disease'].replace('1, 5', '1')
df['Heart_disease'] = df['Heart_disease'].replace('5', '1')
df['Heart_disease'] = df['Heart_disease'].replace('1, 2', '1')
df['Heart_disease'] = df['Heart_disease'].replace('2', '1')
df['Heart_disease'] = df['Heart_disease'].replace('1, 3, 5', '1')
df['Heart_disease'] = df['Heart_disease'].replace('1, 4', '1')
df['Heart_disease'] = df['Heart_disease'].replace('4', '1')
df['Heart_disease'] = df['Heart_disease'].replace('3, 5', '1')

In [26]:
df['Heart_disease'].value_counts()

0    1010
1     920
Name: Heart_disease, dtype: int64

In [27]:
#Initial_CEA

Initial_CEA_cate = [5]
df['Initial_CEA'] = np.digitize(df['Initial_CEA'], Initial_CEA_cate)
df

,Age,Sex,ASA,BMI,DM_history,Pulmonary_disease,Liver_disease,Kidney_disease,Heart_disease,Initial_CEA,...,pN,Harvested_LN,Positive_LN,Early_Complication,Postop_Chemotherapy,Postop_Chemo_Regimen,Survival_Status,Death_Date,Last_Follow_Up_Date,Operation_date
2,1,1,3,2,1,0,0,0,0,0,...,31,33,1,0,1,1,1,2007-09-24 00:00:00,2007-09-24 00:00:00,2004-01-08 00:00:00
3,0,2,2,3,0,0,0,0,1,0,...,2,13,0,0,1,1,0,NaN,2019-03-07 00:00:00,2004-01-15 00:00:00
5,1,1,2,1,0,0,0,0,0,0,...,2,27,0,0,1,1,0,NaN,2019-09-05 00:00:00,2004-01-24 00:00:00
6,0,2,3,2,0,0,0,0,1,1,...,2,28,0,0,1,1,0,NaN,2019-12-05 00:00:00,2004-01-26 00:00:00
7,0,1,2,2,0,0,0,0,0,0,...,2,33,0,0,1,1,0,NaN,2018-12-31 00:00:00,2004-01-27 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2087,0,1,2,2,1,0,0,0,0,0,...,41,21,4,0,1,4,0,NaN,2019-10-02 00:00:00,2018-12-17 00:00:00
2088,1,1,2,2,0,1,0,0,1,0,...,42,40,28,0,0,NaN,0,NaN,2019-04-22 00:00:00,2018-12-17 00:00:00
2089,1,2,2,2,0,0,0,0,1,0,...,2,39,0,0,1,3,0,NaN,2019-10-25 00:00:00,2018-12-18 00:00:00
2090,1,1,2,3,1,0,0,1,1,1,...,42,15,7,0,1,3 + 7,0,NaN,2019-11-06 00:00:00,2018-12-19 00:00:00


In [28]:
# LVI

df['LVI'].value_counts()

32    995
31    726
11    165
4      14
12     12
21      9
42      5
41      2
22      1
3       1
Name: LVI, dtype: int64

In [29]:
df['LVI'] = df['LVI'].astype(str)

In [30]:
df['LVI'] = df['LVI'].replace('11', '1')
df['LVI'] = df['LVI'].replace('21', '1')
df['LVI'] = df['LVI'].replace('31', '1')
df['LVI'] = df['LVI'].replace('12', '2')
df['LVI'] = df['LVI'].replace('22', '2')
df['LVI'] = df['LVI'].replace('32', '2')
df['LVI'] = df['LVI'].replace('1, 3, 5', '1')
df['LVI'] = df['LVI'].replace('1, 4', '1')

### 결측치 삭제

In [31]:
df.isna().sum()

Age                               0
Sex                               0
ASA                               0
BMI                               0
DM_history                        0
Pulmonary_disease                 0
Liver_disease                     0
Kidney_disease                    0
Heart_disease                     0
Initial_CEA                       0
Hereditary_colorectal_tumor       0
Primary_recurrent                 0
Perforation                       0
Obstruction                       0
Emergency                         0
Prior_Dx_cancer                   0
pT                                3
pM                                3
pTNM                              3
K-ras                             0
N-ras                             0
BRAF                              0
LVI                               0
PNI                               0
pN                                0
Harvested_LN                      0
Positive_LN                       0
Early_Complication          

In [32]:
#pT

df['pT']=df['pT'].fillna(9999)
df = df[df['pT'] != 9999]

In [33]:
#pM

df['pM']=df['pM'].fillna(9999)
df = df[df['pM'] != 9999]

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until


In [34]:
#Postop_Chemotherapy

df['Postop_Chemotherapy']=df['Postop_Chemotherapy'].fillna(0)

## to_numeric

In [35]:
df['Sex'] = pd.to_numeric(df['Sex'])
df['ASA'] = pd.to_numeric(df['ASA'])
df['DM_history'] = pd.to_numeric(df['DM_history'])
df['Pulmonary_disease'] = pd.to_numeric(df['Pulmonary_disease'])
df['Kidney_disease'] = pd.to_numeric(df['Kidney_disease'])
df['LVI'] = pd.to_numeric(df['LVI'])
df['pN'] = pd.to_numeric(df['pN'])
df['Postop_Chemotherapy'] = pd.to_numeric(df['Postop_Chemotherapy'])
df['Heart_disease'] = pd.to_numeric(df['Heart_disease'])

## OS 변수 생성

In [36]:
def overall_survival(df):
  if(pd.isnull(df['Death_Date'])): #생존
    return df['Last_Follow_Up_Date'] - df['Operation_date']
  else:
    return df['Death_Date'] - df['Operation_date']


In [37]:
from datetime import datetime
df['OS'] = df.apply(overall_survival, axis = 1)
df['OS'] = df['OS'].dt.days

In [38]:
df['OS'] = round(df['OS']/30,2)

In [39]:
df['OS']

2        45.17
3       184.33
5       190.10
6       193.07
7       181.73
         ...  
2087      9.63
2088      4.20
2089     10.37
2090     10.73
2091      1.07
Name: OS, Length: 1927, dtype: float64

### 5년 기준 범주화

In [40]:
OS_cate = [60]
df['OS_5year'] = np.digitize(df['OS'], OS_cate)

In [41]:
df['OS_5year'].value_counts()

0    1148
1     779
Name: OS_5year, dtype: int64

## ANN

In [42]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras.models import Sequential
from keras.layers import Dense
from sklearn.metrics import confusion_matrix

In [43]:
target_col = 'OS_5year'
target = df[target_col]
target=target.astype('int')
features = df[['Age', 'Sex', 'ASA', 'BMI', 'DM_history', 'Pulmonary_disease','Kidney_disease', 'Heart_disease', 'Initial_CEA', 'Perforation', 'Obstruction',
               'pT','pM', 'LVI', 'PNI', 'pN', 'Postop_Chemotherapy']]
target.value_counts()
features.value_counts()

Age  Sex  ASA   BMI  DM_history  Pulmonary_disease  Kidney_disease  Heart_disease  Initial_CEA  Perforation  Obstruction  pT  pM  LVI  PNI  pN  Postop_Chemotherapy
1    2    2     2    0           0                  0               0              0            0            0            6   1   2    2    2   0                      9
                                                                    1              0            0            0            6   1   2    2    2   0                      9
0    1    2     2    0           0                  0               0              0            0            0            6   1   2    2    2   0                      8
1    1    2     3    0           0                  0               1              0            0            0            6   1   2    2    2   0                      8
                2    1           0                  0               1              0            0            0            5   1   2    2    2   0               

In [44]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size = 0.4, random_state=0
)

In [45]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [46]:
classifier = Sequential()
classifier.add(Dense(activation = "relu", input_dim = 17, 
                     units = 8, kernel_initializer = "uniform"))
classifier.add(Dense(activation = "relu", units = 14, 
                     kernel_initializer = "uniform"))
classifier.add(Dense(activation = "sigmoid", units = 1, 
                     kernel_initializer = "uniform"))
classifier.compile(optimizer = 'adam' , loss = 'binary_crossentropy', 
                   metrics = ['accuracy'] )

In [47]:
classifier.fit(X_train , y_train , batch_size = 8 ,epochs = 100  )


Epoch 1/100
145/145 [==============================] - 1s 2ms/step - loss: 0.6809 - accuracy: 0.5848
Epoch 2/100
145/145 [==============================] - 0s 2ms/step - loss: 0.6122 - accuracy: 0.6773
Epoch 3/100
145/145 [==============================] - 1s 3ms/step - loss: 0.5788 - accuracy: 0.7016
Epoch 4/100
145/145 [==============================] - 0s 3ms/step - loss: 0.5685 - accuracy: 0.7016
Epoch 5/100
145/145 [==============================] - 0s 3ms/step - loss: 0.5640 - accuracy: 0.7145
Epoch 6/100
145/145 [==============================] - 1s 3ms/step - loss: 0.5612 - accuracy: 0.7093
Epoch 7/100
145/145 [==============================] - 1s 5ms/step - loss: 0.5584 - accuracy: 0.7215
Epoch 8/100
145/145 [==============================] - 1s 4ms/step - loss: 0.5566 - accuracy: 0.7145
Epoch 9/100
145/145 [==============================] - 1s 5ms/step - loss: 0.5542 - accuracy: 0.7163
Epoch 10/100
145/145 [==============================] - 1s 4ms/step - loss: 0.5522 - accura

In [48]:
y_pred = classifier.predict(X_test)
y_pred = (y_pred > 0.5)

In [49]:
cm = confusion_matrix(y_test,y_pred)
cm

array([[352, 120],
       [133, 166]])

In [50]:
accuracy = (cm[0][0]+cm[1][1])/(cm[0][1] + cm[1][0] +cm[0][0] +cm[1][1])
print(accuracy*100)

67.18547341115433
